# FAR tests using different datasets

In [1]:
import sys

!{sys.executable} -m pip install shap
!{sys.executable} -m pip install nbimporter

In [2]:
import numpy as np
import pandas as pd
import sklearn as skl
import zipfile as zf
import matplotlib.pyplot as plt
import nbimporter
import shap

import Ft_Att_Rank as far

# Iris dataset

In [3]:
import sklearn.datasets

iris= sklearn.datasets.load_iris()

X_iris= iris.data
Y_iris= iris.target

In [4]:
# convert iris to a df and drop the setosa rows
df_iris= pd.DataFrame(data= np.c_[X_iris, Y_iris], columns= iris['feature_names'] + ['target'])
df_iris= df_iris[df_iris['target']!= 0].reset_index(drop= True)

In [5]:
# split df_iris into features (x) and target (y)
df_iris_x= df_iris.loc[:,df_iris.columns[0:4]]
df_iris_y= df_iris.loc[:,df_iris.columns[4:5]]

df_iris_x= far.normalize_selected_cols(df_iris_x,df_iris_x.columns)

df_iris_x.shape

(100, 4)

In [6]:
# ML model - Random forest
import sklearn.ensemble

train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(df_iris_x,df_iris_y,train_size=0.80,random_state=1234)

rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)
rf.fit(train, labels_train.values.ravel())
rf_acc= sklearn.metrics.accuracy_score(labels_test, rf.predict(test))

rf_acc

0.85

In [8]:
# ML model - XGBoost random forest
import xgboost as xgb

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',verbosity=0)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.85

In [9]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train.values.ravel(), 
                                          labels_test.values.ravel(), repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train.values.ravel(),
                                               labels_test.values.ravel(), repeat_train)

In [10]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [11]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    4.597701e-01
petal width (cm)     4.022989e-01
sepal length (cm)    1.379310e-01
sepal width (cm)     3.406366e-17
dtype: float64

In [12]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.389768
petal width (cm)     0.300783
sepal length (cm)    0.174804
sepal width (cm)     0.129429
dtype: float64

In [13]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    0.353756
petal width (cm)     0.287302
sepal length (cm)    0.197763
sepal width (cm)     0.153875
dtype: float64

In [14]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

petal length (cm)    4.600998e-01
petal width (cm)     4.276808e-01
sepal length (cm)    1.122195e-01
sepal width (cm)    -1.732514e-17
dtype: float64

# Wine dataset

In [14]:
wine= pd.read_csv('datasets/wine.data',header=None)

wine.columns= ['target','alcohol','malicAcid','ash','ashalcalinity','magnesium','totalPhenols','flavanoids','nonFlavanoidPhenols','proanthocyanins',
               'colorIntensity','hue','od280_od315','proline']

wine= wine[wine['target']!= 3].reset_index(drop= True)

x_wine= wine.iloc[:,1:len(wine.columns)].copy()
y_wine= np.asarray(wine['target'])

x_wine.shape

(130, 13)

In [15]:
x_wine= far.pre_proc_fillna_num_fts(x_wine, x_wine.columns, num_type='mean')

x_wine= far.normalize_selected_cols(x_wine, x_wine.columns)

In [16]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_wine,y_wine,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss')
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].


1.0

In [17]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [18]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [19]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         3.333333e-01
flavanoids             2.083333e-01
alcohol                2.083333e-01
proline                1.666667e-01
od280_od315            8.333333e-02
hue                    5.681174e-17
proanthocyanins        5.574054e-17
nonFlavanoidPhenols    2.380877e-17
totalPhenols           1.165494e-17
malicAcid              8.400598e-19
ashalcalinity         -6.213089e-18
ash                   -1.153291e-17
magnesium             -3.030033e-17
dtype: float64

In [20]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         0.202925
proline                0.132574
alcohol                0.128071
flavanoids             0.128071
od280_od315            0.065017
nonFlavanoidPhenols    0.041357
malicAcid              0.041357
magnesium              0.041357
totalPhenols           0.041357
ash                    0.041357
proanthocyanins        0.041357
ashalcalinity          0.041357
hue                    0.041357
dtype: float64

In [21]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         0.204238
proline                0.134571
alcohol                0.126691
flavanoids             0.126691
od280_od315            0.070039
proanthocyanins        0.040705
hue                    0.040705
ash                    0.040705
nonFlavanoidPhenols    0.040705
totalPhenols           0.040705
ashalcalinity          0.040705
malicAcid              0.040705
magnesium              0.040705
dtype: float64

In [22]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

colorIntensity         3.476821e-01
proline                2.516556e-01
alcohol                1.655629e-01
flavanoids             1.655629e-01
od280_od315            6.953642e-02
hue                    5.074904e-17
proanthocyanins        4.983179e-17
nonFlavanoidPhenols    2.506096e-17
magnesium              1.806503e-17
ash                    5.521896e-18
ashalcalinity          2.622769e-18
totalPhenols          -5.860638e-18
malicAcid             -8.355892e-18
dtype: float64

# Titanic dataset

In [23]:
# https://www.kaggle.com/c/titanic
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [24]:
X_train= far.pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')
X_train= far.pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train= far.normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= far.normalize_selected_cols(X_train_ohe, numeric_columns)

X_train.shape

(891, 7)

In [25]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train.values.ravel())
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.8435754189944135

In [26]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,numeric_columns,num_type='mean',cat_type='median')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [27]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [28]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [29]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

Age           2.436283e-01
Sex_female    1.482457e-01
Sex_male      1.445947e-01
Embarked_S    1.032141e-01
Fare          8.712136e-02
Parch         7.103658e-02
Pclass_3      7.032120e-02
Pclass_1      6.763143e-02
SibSp         3.950060e-02
Embarked_C    1.358258e-02
Embarked_Q    1.112345e-02
Pclass_2     -1.257455e-16
dtype: float64

In [30]:
far.ft_importance_df(stationary_d1,train.columns,rp_list)

,Feature,Importance
0,Sex,0.292840
1,Age,0.243628
2,Pclass,0.137953
3,Embarked,0.127920
4,Fare,0.087121
5,Parch,0.071037
6,SibSp,0.039501


In [31]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.216659
Sex_male      0.215148
Age           0.096492
Fare          0.082576
Embarked_S    0.066227
SibSp         0.053928
Embarked_C    0.053607
Pclass_1      0.049855
Pclass_3      0.046073
Embarked_Q    0.045411
Pclass_2      0.043312
Parch         0.030541
dtype: float64

In [32]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

,Feature,Importance
0,Sex,0.431807
1,Embarked,0.165245
2,Pclass,0.139241
3,Age,0.096492
4,Fare,0.082576
5,SibSp,0.053928
6,Parch,0.030541


In [33]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.239007
Sex_male      0.236512
Age           0.094620
Fare          0.076875
Embarked_S    0.060828
Embarked_Q    0.048471
SibSp         0.044467
Embarked_C    0.042309
Pclass_3      0.041977
Pclass_1      0.040352
Parch         0.038350
Pclass_2      0.036115
dtype: float64

In [34]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

,Feature,Importance
0,Sex,0.431807
1,Embarked,0.165245
2,Pclass,0.139241
3,Age,0.096492
4,Fare,0.082576
5,SibSp,0.053928
6,Parch,0.030541


In [35]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

Age           3.106063e-01
Embarked_S    1.237700e-01
Fare          1.084219e-01
Pclass_3      9.875373e-02
Parch         8.652123e-02
Sex_female    7.578856e-02
Sex_male      7.372754e-02
Pclass_1      5.057279e-02
Embarked_Q    3.832171e-02
SibSp         2.523268e-02
Embarked_C    8.283555e-03
Pclass_2      6.898985e-17
dtype: float64

In [36]:
far.ft_importance_df(stationary_d4,train.columns,rp_list)

,Feature,Importance
0,Age,0.310606
1,Embarked,0.170375
2,Sex,0.149516
3,Pclass,0.149327
4,Fare,0.108422
5,Parch,0.086521
6,SibSp,0.025233


# Heartrisk dataset

In [37]:
# https://www.kaggle.com/pritsheta/heart-attack
ds= zf.ZipFile('datasets/heart.zip')

heart_data= pd.read_csv(ds.open('heart.csv'))

x_heart= heart_data.iloc[:,:(len(heart_data.columns)-1)].copy()
y_heart= np.asarray(heart_data['target'])

x_heart.shape

(303, 13)

In [38]:
x_heart= far.pre_proc_fillna_num_fts(x_heart, x_heart.columns,num_type='mean')

x_heart= far.normalize_selected_cols(x_heart, x_heart.columns)

In [39]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_heart,y_heart,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.7377049180327869

In [40]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [41]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [42]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

ca          0.181568
cp          0.149432
thalach     0.138426
exang       0.085205
thal        0.080144
oldpeak     0.076793
chol        0.076766
slope       0.068077
sex         0.043236
trestbps    0.036708
age         0.036187
restecg     0.020251
fbs         0.007206
dtype: float64

In [43]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          0.180383
sex         0.133823
slope       0.122023
oldpeak     0.088924
fbs         0.076804
ca          0.075208
trestbps    0.061086
chol        0.054086
restecg     0.054035
age         0.049418
thal        0.037397
exang       0.036421
thalach     0.030393
dtype: float64

In [44]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

sex         0.140201
cp          0.125884
slope       0.112818
ca          0.084262
oldpeak     0.078196
fbs         0.066764
chol        0.064497
trestbps    0.060199
restecg     0.059374
age         0.054657
thal        0.051992
exang       0.051441
thalach     0.049713
dtype: float64

In [45]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

cp          0.211618
thalach     0.136147
ca          0.111288
thal        0.086175
exang       0.085089
slope       0.082586
oldpeak     0.080948
sex         0.076193
chol        0.075240
trestbps    0.019580
age         0.015581
restecg     0.015322
fbs         0.004233
dtype: float64

# Breast cancer Wisconsin dataset

In [46]:
# https://www.kaggle.com/uciml/breast-cancer-wisconsin-data
wdbc= sklearn.datasets.load_breast_cancer()

X_wb_cancer= pd.DataFrame(wdbc.data,columns=wdbc.feature_names)
Y_wb_cancer= wdbc.target

X_wb_cancer.shape

(569, 30)

In [47]:
X_wb_cancer= far.pre_proc_fillna_num_fts(X_wb_cancer,wdbc.feature_names,num_type='mean')

X_wb_cancer= far.normalize_selected_cols(X_wb_cancer, wdbc.feature_names)

In [48]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(X_wb_cancer,Y_wb_cancer,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.9473684210526315

In [49]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [50]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [51]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              1.450848e-01
mean texture               1.106096e-01
worst area                 9.071237e-02
worst concavity            8.893382e-02
worst perimeter            7.644444e-02
area error                 6.932635e-02
worst radius               6.771069e-02
worst concave points       6.607988e-02
mean concave points        5.593596e-02
mean smoothness            3.787910e-02
mean area                  2.887780e-02
mean concavity             2.168325e-02
mean radius                2.109623e-02
worst smoothness           2.063137e-02
worst fractal dimension    1.600510e-02
concavity error            1.005542e-02
mean compactness           8.904488e-03
radius error               8.082448e-03
worst compactness          7.257570e-03
symmetry error             7.257570e-03
worst symmetry             7.257570e-03
mean perimeter             7.121696e-03
texture error              5.949683e-03
perimeter error            5.426599e-03
mean fractal dimension     5.124805e-03


In [52]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

mean smoothness            0.122922
worst area                 0.115768
worst concave points       0.094101
mean concave points        0.056460
mean texture               0.044780
worst texture              0.039820
worst concavity            0.036076
mean radius                0.030334
worst compactness          0.028472
radius error               0.026622
area error                 0.026114
mean perimeter             0.026019
mean concavity             0.025093
fractal dimension error    0.024860
worst perimeter            0.023453
concavity error            0.022829
worst radius               0.021633
mean symmetry              0.021302
worst symmetry             0.021233
symmetry error             0.021233
perimeter error            0.021006
worst fractal dimension    0.020746
texture error              0.019944
smoothness error           0.018402
compactness error          0.018169
concave points error       0.018169
mean compactness           0.017386
mean fractal dimension     0

In [53]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

mean smoothness            0.125663
worst area                 0.108075
worst concave points       0.093324
mean concave points        0.057461
mean texture               0.048207
worst texture              0.037662
worst concavity            0.034261
mean radius                0.030394
worst radius               0.027785
area error                 0.027288
mean perimeter             0.025921
worst perimeter            0.025211
mean concavity             0.024528
worst compactness          0.023533
radius error               0.022780
fractal dimension error    0.021840
concavity error            0.020715
perimeter error            0.020438
worst fractal dimension    0.020259
symmetry error             0.020140
worst symmetry             0.020140
texture error              0.019835
mean symmetry              0.019624
compactness error          0.018889
concave points error       0.018889
mean compactness           0.018741
smoothness error           0.018592
mean fractal dimension     0

In [54]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

worst texture              1.268658e-01
area error                 1.223114e-01
mean texture               1.091927e-01
worst radius               9.205973e-02
mean smoothness            8.425856e-02
worst area                 7.758575e-02
worst concave points       7.541681e-02
mean concave points        6.409379e-02
mean area                  4.970459e-02
worst perimeter            4.741159e-02
worst concavity            4.530037e-02
worst smoothness           1.647827e-02
mean radius                1.188608e-02
mean concavity             1.175112e-02
worst fractal dimension    9.264057e-03
mean compactness           6.667604e-03
mean perimeter             5.074544e-03
texture error              4.714359e-03
mean fractal dimension     4.625873e-03
concavity error            4.549697e-03
radius error               4.415574e-03
perimeter error            4.382246e-03
worst compactness          4.327088e-03
worst symmetry             4.327088e-03
symmetry error             4.327088e-03


# Heart failure prediction dataset

In [55]:
# https://www.kaggle.com/fedesoriano/heart-failure-prediction
ds= zf.ZipFile('datasets/heart_2.zip')

heart_2_data= pd.read_csv(ds.open('heart.csv'))

x_heart2= heart_2_data.iloc[:,:(len(heart_2_data.columns)-1)].copy()
y_heart2= np.asarray(heart_2_data['HeartDisease'])

x_heart2.shape

(918, 11)

In [56]:
numeric_columns= ['Age','RestingBP','Cholesterol','MaxHR','Oldpeak']
categor_columns= list(filter(lambda x:x not in numeric_columns,x_heart2.columns))

x_heart2= far.pre_proc_fillna_num_fts(x_heart2,numeric_columns,num_type='median')
x_heart2= far.pre_proc_fillna_cat_fts(x_heart2,categor_columns,cat_type='mode')

Sex= [1 if s == 'M' else 0 for s in x_heart2['Sex']]
ExerciseAngina= [1 if e == 'Y' else 0 for e in x_heart2['ExerciseAngina']]

x_heart2['Sex']= Sex
x_heart2['ExerciseAngina']= ExerciseAngina

x_heart2_ohe= pd.get_dummies(x_heart2,columns=['ChestPainType','RestingECG','ST_Slope'])
x_heart2_ohe= far.normalize_selected_cols(x_heart2_ohe, numeric_columns)

x_heart2_ohe.shape

(918, 18)

In [57]:
# ML model - XGBoost random forest
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(x_heart2_ohe,y_heart2,train_size=0.80,random_state=1234)

xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)
xgb_model.fit(train, labels_train)
xgb_acc= sklearn.metrics.accuracy_score(labels_test, xgb_model.predict(test))

xgb_acc

0.8913043478260869

In [58]:
# Feature Attribution using Raking
repeat_train= 1
num_fts= len(train.columns)

acc_all_fts= far.train_model_get_acc_mean(xgb_model, train, test, labels_train, labels_test, repeat_train)

replace_ft= far.replace_values(train,train.columns,num_type='mean')

acc_no_i, acc_no_ij= far.remove_and_retrain_v1(xgb_model, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [59]:
# get the probability matrix
p_matrix1= far.get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= far.get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= far.get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= far.get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

# get the stationary distribution
stationary_d1= far.p_matrix_to_stationary_dist(p_matrix1)
stationary_d2= far.p_matrix_to_stationary_dist(p_matrix2)
stationary_d3= far.p_matrix_to_stationary_dist(p_matrix3)
stationary_d4= far.p_matrix_to_stationary_dist(p_matrix4)

In [60]:
rp_list= {'ChestPainType_ASY':'ChestPainType','ChestPainType_ATA':'ChestPainType','ChestPainType_NAP':'ChestPainType','ChestPainType_TA':'ChestPainType',
          'RestingECG_LVH':'RestingECG','RestingECG_Normal':'RestingECG','RestingECG_ST':'RestingECG',
          'ST_Slope_Down':'ST_Slope','ST_Slope_Flat':'ST_Slope','ST_Slope_Up':'ST_Slope'}

In [61]:
ft_ranking= pd.Series(stationary_d1, index=train.columns).sort_values(ascending=False)
ft_ranking

Oldpeak              1.324297e-01
ST_Slope_Up          1.294279e-01
Cholesterol          1.207097e-01
ChestPainType_ASY    1.117173e-01
RestingBP            9.852778e-02
ST_Slope_Flat        8.554786e-02
ExerciseAngina       7.177764e-02
MaxHR                5.348612e-02
FastingBS            4.710356e-02
Sex                  4.678875e-02
Age                  4.211399e-02
RestingECG_LVH       3.528255e-02
RestingECG_Normal    1.227630e-02
ChestPainType_ATA    6.626210e-03
ChestPainType_TA     3.498051e-03
ChestPainType_NAP    2.600022e-03
RestingECG_ST        8.666738e-05
ST_Slope_Down        4.936462e-17
dtype: float64

In [62]:
far.ft_importance_df(stationary_d1,train.columns,rp_list)

,Feature,Importance
0,ST_Slope,0.214976
1,Oldpeak,0.132430
2,ChestPainType,0.124442
3,Cholesterol,0.120710
4,RestingBP,0.098528
5,ExerciseAngina,0.071778
6,MaxHR,0.053486
7,RestingECG,0.047646
8,FastingBS,0.047104
9,Sex,0.046789


In [63]:
ft_ranking= pd.Series(stationary_d2, index=train.columns).sort_values(ascending=False)
ft_ranking

ST_Slope_Up          0.134080
ChestPainType_NAP    0.096787
ChestPainType_ASY    0.080382
ExerciseAngina       0.077604
ST_Slope_Flat        0.073980
ChestPainType_TA     0.065683
ST_Slope_Down        0.065022
FastingBS            0.055817
MaxHR                0.043516
RestingECG_LVH       0.041438
RestingECG_Normal    0.040374
Oldpeak              0.037981
RestingECG_ST        0.035768
Cholesterol          0.035354
ChestPainType_ATA    0.035102
RestingBP            0.030720
Sex                  0.025854
Age                  0.024290
dtype: float64

In [64]:
far.ft_importance_df(stationary_d2,train.columns,rp_list)

,Feature,Importance
0,ChestPainType,0.277953
1,ST_Slope,0.273083
2,RestingECG,0.117580
3,ExerciseAngina,0.077604
4,FastingBS,0.055817
5,MaxHR,0.043516
6,Oldpeak,0.037981
7,Cholesterol,0.035354
8,RestingBP,0.030720
9,Sex,0.025854


In [65]:
ft_ranking= pd.Series(stationary_d3, index=train.columns).sort_values(ascending=False)
ft_ranking

ChestPainType_NAP    0.107558
ST_Slope_Up          0.106056
ChestPainType_ASY    0.088187
ST_Slope_Flat        0.063925
ExerciseAngina       0.063729
FastingBS            0.061730
MaxHR                0.046565
Oldpeak              0.046023
ChestPainType_TA     0.045024
ST_Slope_Down        0.044266
RestingECG_LVH       0.043770
RestingECG_Normal    0.043210
RestingBP            0.042880
Cholesterol          0.042030
RestingECG_ST        0.041362
ChestPainType_ATA    0.040912
Sex                  0.036499
Age                  0.036156
dtype: float64

In [66]:
far.ft_importance_df(stationary_d3,train.columns,rp_list)

,Feature,Importance
0,ChestPainType,0.281682
1,ST_Slope,0.214247
2,RestingECG,0.128343
3,ExerciseAngina,0.063729
4,FastingBS,0.061730
5,MaxHR,0.046565
6,Oldpeak,0.046023
7,RestingBP,0.042880
8,Cholesterol,0.042030
9,Sex,0.036499


In [67]:
ft_ranking= pd.Series(stationary_d4, index=train.columns).sort_values(ascending=False)
ft_ranking

Oldpeak              1.729517e-01
ChestPainType_ASY    1.378086e-01
Cholesterol          1.262387e-01
ExerciseAngina       9.131846e-02
RestingBP            8.993494e-02
ST_Slope_Up          8.315521e-02
MaxHR                6.449903e-02
FastingBS            6.073327e-02
ST_Slope_Flat        5.439405e-02
Age                  4.002266e-02
Sex                  2.357352e-02
ChestPainType_NAP    2.176364e-02
RestingECG_LVH       1.964042e-02
RestingECG_Normal    7.899765e-03
ChestPainType_ATA    4.411573e-03
ChestPainType_TA     1.279311e-03
RestingECG_ST        3.752353e-04
ST_Slope_Down       -5.758602e-17
dtype: float64

In [68]:
far.ft_importance_df(stationary_d4,train.columns,rp_list)

,Feature,Importance
0,Oldpeak,0.172952
1,ChestPainType,0.165263
2,ST_Slope,0.137549
3,Cholesterol,0.126239
4,ExerciseAngina,0.091318
5,RestingBP,0.089935
6,MaxHR,0.064499
7,FastingBS,0.060733
8,Age,0.040023
9,RestingECG,0.027915
